In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F


@dp.materialized_view(comment="Completed ride revenue aggregated per minute over the last 60 minutes.")
def gold_revenue_per_minute():
    return (
        spark.read.table("silver_ride_events")
        # 1. Explicit cast — guards against string/epoch storage
        .withColumn("event_time", F.col("event_time").cast("timestamp"))
        .filter(F.col("status") == "completed")
        # 2. Timezone-safe comparison using convert_timezone to normalize to UTC
        .filter(
            F.col("event_time") >= F.convert_timezone(None, F.lit("UTC"),
                F.current_timestamp() - F.expr("INTERVAL 60 MINUTES")
            )
        )
        # 3. F.col() — not a raw string — in date_trunc
        .withColumn("minute_bucket", F.date_trunc("minute", F.col("event_time")))
        .groupBy("minute_bucket")
        .agg(
            F.round(F.sum("final_fare"), 2).alias("revenue_inr"),
            F.count("ride_id").alias("completed_rides"),
        )
        .orderBy("minute_bucket")
    )